<a href="https://colab.research.google.com/github/abrishkhanzai441-arch/Program-/blob/main/teamd1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!apt-get install -y tesseract-ocr poppler-utils -q
!pip install pytesseract pdf2image pymupdf -q

from google.colab import files
import pymupdf
import pytesseract
from pdf2image import convert_from_path
import re

database = {}

print("📤 Upload your PDF files...")
uploaded = files.upload()

for filename, file_data in uploaded.items():
    with open(filename, "wb") as f:
        f.write(file_data)

    text = ""
    if filename.lower().endswith(".pdf"):
        doc = pymupdf.open(filename)
        for page in doc:
            text += page.get_text()

        # If standard text extraction yields nothing, run OCR
        if not text.strip():
            print(f"🔍 No text layer found in {filename}. Running OCR...")
            images = convert_from_path(filename)
            for img in images:
                text += pytesseract.image_to_string(img)

    if text.strip():
        database[filename] = text
        print(f"✅ {filename} -> Saved {len(text)} characters to database.")
    else:
        print(f"⚠️ Could not extract text from {filename}.")

def secure_search(question):
    clean_query = re.sub(r'[^\w\s]', '', question).strip().lower()
    print(f"\n❓ Searching for: '{clean_query}'")
    print("-" * 30)

    found = False
    for fname, content in database.items():
        pos = content.lower().find(clean_query)
        if pos != -1:
            preview = content[max(0, pos - 80) : pos + 150]
            print(f"📄 FOUND IN: {fname}\nAnswer: ...{preview}...\n")
            found = True

    if not found:
        print("❌ Not found in your documents.")

# Test search
print(database.get("liquify effect (2).pdf", "File not found"))

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 212 kB of archives.
After this operation, 735 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 poppler-utils amd64 24.02.0-1ubuntu9.9 [212 kB]
Fetched 212 kB in 0s (462 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
📤 Upload your PDF files...


Saving Penetration_Testing_Assignment.pdf to Penetration_Testing_Assignment.pdf
✅ Penetration_Testing_Assignment.pdf -> Saved 816 characters to database.
File not found


In [7]:
!pip install -q flask flask-cors PyMuPDF sentence-transformers faiss-cpu gradio

In [8]:
import fitz
import gradio as gr
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load AI Model - this is your AI Team's RAG model
print("Loading AI model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

documents = []
doc_names = []
index = faiss.IndexFlatL2(384)

# Function 1: Upload and Process - This is Python Team's main work
def process_upload(files):
    global documents, doc_names, index
    if files is None:
        return "Please upload a file first!"

    count = 0
    for file in files:
        filename = file.name.split("/")[-1]
        text = ""
        try:
            if filename.endswith('.pdf'):
                doc = fitz.open(file.name)
                for page in doc:
                    text += page.get_text()
            else:
                with open(file.name, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()

            if text.strip() == "":
                continue

            documents.append(text)
            doc_names.append(filename)
            vector = model.encode([text])
            index.add(np.array(vector).astype('float32'))
            count += 1
        except Exception as e:
            print(f"Error in {filename}: {e}")

    return f"✅ Successfully processed {count} files: {', '.join(doc_names[-count:])}"

# Function 2: Ask Question - This is RAG
def ask_question(question):
    if len(documents) == 0:
        return "❌ No documents uploaded yet. Please upload files first.", "No source"

    if not question.strip():
        return "Please type a question.", ""

    q_vector = model.encode([question])
    D, I = index.search(np.array(q_vector).astype('float32'), k=1)

    best_doc_index = I[0][0]
    # Handle case where index is empty or invalid
    if best_doc_index < 0 or best_doc_index >= len(documents):
        return "Could not find relevant doc.", ""

    context = documents[best_doc_index]
    source_file = doc_names[best_doc_index]

    # This is where AI gives answer based ONLY on your files (Secure)
    # For demo, we show top 800 chars of context
    answer = f"ANSWER FOUND in '{source_file}':\n\n{context[:800]}...\n\n[This is secure - answer is only from your uploaded files, not from internet]"

    return answer, source_file

# Function 3: API for your Website
def api_for_website(file, question):
    # This same logic will be called by your Web Dev Team's website
    if file is not None:
        process_upload([file])
    if question:
        ans, src = ask_question(question)
        return ans
    return "File uploaded. Now ask a question."

Loading AI model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


In [12]:
# Build the UI - This is what Web Dev Team will see
with gr.Blocks(title="AI Secure Document Portal") as demo:
    gr.Markdown("# 🔐 AI-Powered Secure Document Search Portal")
    gr.Markdown("**Python Team Backend - Ready to link with Web Frontend**")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(label="Upload Private Documents (PDF/TXT)", file_count="multiple")
            upload_btn = gr.Button("Process Documents")
            upload_output = gr.Textbox(label="Status")

        with gr.Column():
            question_input = gr.Textbox(label="Ask question about documents", placeholder="e.g. What is in the salary file?")
            ask_btn = gr.Button("Ask AI")
            answer_output = gr.Textbox(label="AI Answer", lines=10)
            source_output = gr.Textbox(label="Source File")

    upload_btn.click(fn=process_upload, inputs=file_input, outputs=upload_output)
    ask_btn.click(fn=ask_question, inputs=question_input, outputs=[answer_output, source_output])

# This gives you the link for your website!
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/03 20:22:09 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None


In [13]:
!pip install gradio -q

import gradio as gr
import numpy as np

def answer_question(question):
    q_vector = ai_model.encode([question])
    D, I = index.search(np.array(q_vector), 1)
    idx = I[0][0]
    return all_chunks[idx]

# This gives you a permanent public link - no token needed!
demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(label="Your Question"),
    outputs=gr.Textbox(label="Answer"),
    title="My AI Bot API"
)

demo.launch(share=True, server_name="0.0.0.0")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/03 20:25:09 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


<IPython.core.display.Javascript object>

In [14]:
!pip install gradio -q
import gradio as gr
import numpy as np

def answer_question(question):
    if len(documents) == 0:
        return "No documents uploaded yet. Please upload files in the previous cell first."
    if not question.strip():
        return "Please enter a question."

    # Use 'model' (from cell [8]) and handle 2D array output properly
    q_vector = model.encode([question])
    D, I = index.search(np.array(q_vector).astype('float32'), 1)

    idx = I[0][0]
    if idx < 0 or idx >= len(documents):
        return "No relevant answer found."

    return documents[idx]

# Interface setup
demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(label="Your Question"),
    outputs=gr.Textbox(label="Answer"),
    title="My AI Bot API"
)

# Launch locally inside Colab (share=False avoids network timeouts)
demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [16]:
!pip install gradio -q
import gradio as gr
import numpy as np

def api_pipeline(files, question):
    # Process files if uploaded
    if files is not None:
        process_upload(files)

    # Check if documents exist
    if len(documents) == 0:
        return "No documents uploaded yet. Please upload files above first."

    if not question.strip():
        return "Please enter a question."

    # Search for answer
    q_vector = model.encode([question])
    D, I = index.search(np.array(q_vector).astype('float32'), 1)

    idx = I[0][0]
    if idx < 0 or idx >= len(documents):
        return "No relevant answer found."

    return documents[idx]

demo = gr.Interface(
    fn=api_pipeline,
    inputs=[
        gr.File(label="Upload Documents (PDF/TXT)", file_count="multiple"),
        gr.Textbox(label="Your Question")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="My AI Bot API"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5c57cecad471d9e79c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> None
Killing tunnel 127.0.0.1:7863 <> https://5c57cecad471d9e79c.gradio.live


In [ ]:
# FIRST - Install required libraries (This was missing)
!pip install -q gradio PyPDF2

import gradio as gr
from PyPDF2 import PdfReader

# Global memory
stored_text = ""
stored_chunks = []

def ingest(files):
    global stored_text, stored_chunks
    if not files:
        return "❌ Please upload files first."

    # files can be single or list
    if not isinstance(files, list):
        files = [files]

    full_text = ""
    for f in files:
        file_path = f.name if hasattr(f, 'name') else f
        try:
            reader = PdfReader(file_path)
            for page in reader.pages:
                txt = page.extract_text()
                if txt:
                    full_text += txt + "\n"
        except Exception as e:
            return f"Error reading file {file_path}: {str(e)}"

    stored_text = full_text
    # Make chunks for better search
    stored_chunks = [full_text[i:i+1000] for i in range(0, len(full_text), 1000)]

    return f"✅ SUCCESS! Indexed {len(files)} file(s).\nTotal Characters: {len(full_text)}\nChunks: {len(stored_chunks)}\n\nNow go to Tab 2 and ask questions."

def ask(question):
    global stored_text, stored_chunks
    if not stored_text:
        return "❌ No document uploaded! Please go to Tab 1 and click 'Ingest Documents' first."
    if not question or question.strip() == "":
        return "Please type a question."

    q_words = [w.lower() for w in question.split() if len(w) > 2]

    # Find relevant chunks
    results = []
    for chunk in stored_chunks:
        chunk_lower = chunk.lower()
        score = sum(1 for w in q_words if w in chunk_lower)
        if score > 0:
            results.append((score, chunk))

    # Sort by relevance
    results.sort(key=lambda x: x[0], reverse=True)

    if results:
        top_answers = "\n\n---\n\n".join([r[1][:800] for r in results[:3]])
        return f"Found answer for: '{question}'\n\n{top_answers}"
    else:
        return f"No direct keyword match for '{question}'.\n\nHere is document preview:\n{stored_text[:1500]}"

# Build UI with 2 separate functions - EXACTLY what your developer wants
with gr.Blocks(title="Doc Bot - Permanent API") as demo:
    gr.Markdown("# 📄 AI Doc Search - 2 Separate APIs")
    gr.Markdown("**For Website Integration: 1) Ingest first, 2) Then Ask**")

    with gr.Tab("1. Upload / Ingest"):
        files_input = gr.File(label="Upload PDFs", file_count="multiple", file_types=[".pdf"])
        ingest_btn = gr.Button("🔥 Ingest Documents", variant="primary")
        ingest_output = gr.Textbox(label="Status", lines=5)
        ingest_btn.click(fn=ingest, inputs=files_input, outputs=ingest_output)

    with gr.Tab("2. Chat / Ask"):
        question_input = gr.Textbox(label="Your Question", placeholder="e.g. What is this document about?")
        ask_btn = gr.Button("💬 Ask", variant="primary")
        answer_output = gr.Textbox(label="Answer", lines=10)
        ask_btn.click(fn=ask, inputs=question_input, outputs=answer_output)

demo.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.0 MB/s eta 0:00:00
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fcce0fe2407f6a6ecf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
